<a href="https://colab.research.google.com/github/ashokmulchandani/Fine_tuning-ML-Pipleine--Synthetic_Data-Ashok-1/blob/main/Phase_5_Instruction_Fine_Tuning_(SFT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl


In [11]:
# FIRST CELL — Enable GPU (Runtime → Change runtime type → T4 GPU)
# Then verify:
!nvidia-smi

Mon Jul 20 09:22:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |    1929MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:

# Method 1: Manual (for small datasets)
instruction_data = [
    {
        "instruction": "What is Metformin?",
        "input": "",
        "output": "Metformin is a biguanide antihyperglycemic agent used to treat type 2 diabetes."
    },
    {
        "instruction": "How does Metformin work?",
        "input": "",
        "output": "It decreases hepatic glucose production and improves insulin sensitivity."
    },
    {
        "instruction": "What are common side effects?",
        "input": "",
        "output": "Common side effects include nausea, vomiting, diarrhea, and abdominal discomfort."
    },
    {
        "instruction": "When is Metformin contraindicated?",
        "input": "",
        "output": "Metformin is contraindicated in patients with severe renal impairment, acute metabolic acidosis, or hypersensitivity to the drug."
    },
    {
        "instruction": "What class of drug is Metformin?",
        "input": "",
        "output": "Metformin belongs to the biguanide class of antihyperglycemic agents."
    },
]
print(f"✅ {len(instruction_data)} examples ready")

# Method 2: Auto-generate with GPT-4o (Phase 5b covers synthetic data)
# Feed chunk → GPT-4o → "Generate 5 Q&A pairs from this text" → JSON output

✅ 5 examples ready


In [18]:
def format_instruction(example):
    if example["input"]:
        text = "Below is an instruction and input. Write a response.\n\n"
        text += "### Instruction:\n" + example["instruction"] + "\n\n"
        text += "### Input:\n" + example["input"] + "\n\n"
        text += "### Response:\n" + example["output"]
    else:
        text = "Below is an instruction. Write a response.\n\n"
        text += "### Instruction:\n" + example["instruction"] + "\n\n"
        text += "### Response:\n" + example["output"]
    return {"text": text}


In [19]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16)
model.config.use_cache = False

# Optional: If you uploaded Phase 4 adapter, add:
# from peft import PeftModel
# model = PeftModel.from_pretrained(model, "./medical_domain_adapter")
# model = model.merge_and_unload()  # Golden rule! Phase 3.10

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_proj","v_proj"], bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

NameError: name 'model_name' is not defined

In [5]:

# Feed each chunk to GPT-4o
import openai

def generate_qa_pairs(chunk_text):
    prompt = f"""Given this medical text, generate 5 question-answer pairs
in JSON format. Questions should test understanding of the content.

Text: {chunk_text}

Output as JSON list: [{{"instruction": "...", "output": "..."}}]"""

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)

# 10 chunks × 5 Q&A each = 50 training examples
# Cost: ~$0.50 total

In [6]:
!pip install -U bitsandbytes


In [7]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,    # ← must use this, not load_in_8bit=True
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_proj","v_proj"], bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [8]:
!pip install -U pyarrow datasets trl


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

training_args = TrainingArguments(
    output_dir="./instruction_tuned", num_train_epochs=10,
    per_device_train_batch_size=4, learning_rate=2e-4,
    fp16=True, logging_steps=5,
)

trainer = SFTTrainer(
    model=model, train_dataset=dataset, tokenizer=tokenizer,
    max_seq_length=512, formatting_func=format_instruction,
    args=training_args,
)

trainer.train()
print("✅ Training complete!")

NameError: name 'dataset' is not defined

In [23]:
# ===== CELL 1: Install (run this first in its own cell) =====
# !pip install -q transformers datasets peft bitsandbytes accelerate trl

# ===== EVERYTHING ELSE in ONE cell =====

# Step 1: Define instruction data
instruction_data = [
    {"instruction": "What is Metformin?", "input": "", "output": "Metformin is a biguanide antihyperglycemic agent used to treat type 2 diabetes mellitus."},
    {"instruction": "How does Metformin work?", "input": "", "output": "It decreases hepatic glucose production and improves insulin sensitivity."},
    {"instruction": "What are common side effects?", "input": "", "output": "Common side effects include nausea, vomiting, diarrhea, and abdominal discomfort."},
    {"instruction": "When is Metformin contraindicated?", "input": "", "output": "Metformin is contraindicated in patients with severe renal impairment, acute metabolic acidosis, or hypersensitivity to the drug."},
    {"instruction": "What class of drug is Metformin?", "input": "", "output": "Metformin belongs to the biguanide class of antihyperglycemic agents."},
]
print(f"✅ {len(instruction_data)} examples ready")

# Step 2: Format function
def format_instruction(example):
    if example["input"]:
        text = "Below is an instruction and input. Write a response.\n\n"
        text += "### Instruction:\n" + example["instruction"] + "\n\n"
        text += "### Input:\n" + example["input"] + "\n\n"
        text += "### Response:\n" + example["output"]
    else:
        text = "Below is an instruction. Write a response.\n\n"
        text += "### Instruction:\n" + example["instruction"] + "\n\n"
        text += "### Response:\n" + example["output"]
    return {"text": text}

# Step 3: Tokenize
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = Dataset.from_list(instruction_data)
dataset = dataset.map(format_instruction)

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
print(f"✅ Dataset ready: {len(dataset)} examples")

# Step 4: Load model + LoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_proj","v_proj"], bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Step 5: Train
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./instruction_tuned", num_train_epochs=10,
    per_device_train_batch_size=4, learning_rate=2e-4,
    logging_steps=5,
)


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=format_instruction,
    args=training_args,
)



trainer.train()
print("✅ Training complete!")

# Step 6: Test
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = "### Instruction:\nWhat is Metformin?\n\n### Response:\n"
result = pipe(prompt, max_new_tokens=80, temperature=0.7)
print(result[0]["generated_text"])


✅ 5 examples ready


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ Dataset ready: 5 examples


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Step,Training Loss
5,3.257775
10,1.264192
15,0.574804
20,0.433350


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


✅ Training complete!


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
[transformers] Ignoring clean_up_tokenization_sp

### Instruction:
What is Metformin?

### Response:
Metrics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')